# Geometric Brownian Motion Demo

Translated from QMCJu's GBM/gbm_demo.ipynb

Demonstrates GeometricBrownianMotion for financial modeling:
  S(t) = S₀ exp[(γ - σ²/2) t + σ W(t)]

where W(t) is a standard Brownian motion.

In [ ]:
using QMCJu
using Statistics
using Printf

Basic GBM Sample Generation

In [ ]:
println("="^60)
println("GeometricBrownianMotion: Basic Usage")
println("="^60)

d = 4           # 4 time steps
S0 = 100.0      # initial price
γ = 0.05        # drift (5% annual return)
σ2 = 0.04       # diffusion (volatility² = 20% vol)

dd = DigitalNetB2(d; seed=7)
gbm = GeometricBrownianMotion(dd;
    t_final=1.0, initial_value=S0, drift=γ, diffusion=σ2)

println("  $gbm")
println("  Time vector: t = [$(join([@sprintf("%.2f", t) for t in gbm.time_vector], ", "))]")
println()

n = 10_000
x = gen_samples(dd, n)
paths = transform(gbm, x)

println("  Generated $n sample paths")
println("  Path shape: $(size(paths))")
println()

Show a few sample paths

In [ ]:
println("  First 5 paths (S₀, S(t₁), S(t₂), S(t₃), S(T)):")
for i in 1:5
    @printf("    Path %d: %6.2f → [%6.2f  %6.2f  %6.2f  %6.2f]\n",
            i, S0, paths[i,1], paths[i,2], paths[i,3], paths[i,4])
end
println()

Statistical Properties

In [ ]:
println("="^60)
println("Statistical Properties of GBM")
println("="^60)

E[S(t)] = S₀ exp(γ t)

In [ ]:
for j in 1:d
    t = gbm.time_vector[j]
    exact_mean = S0 * exp(γ * t)
    empirical_mean = mean(paths[:, j])
    @printf("  t = %.2f: E[S(t)] = %.2f (exact = %.2f, err = %.2f%%)\n",
            t, empirical_mean, exact_mean,
            100.0 * abs(empirical_mean - exact_mean) / exact_mean)
end
println()

GBM values must be positive

In [ ]:
@printf("  All values positive: %s (min = %.4f)\n",
        all(paths .> 0) ? "YES" : "NO", minimum(paths))
println()

GBM with FinancialOption

In [ ]:
println("="^60)
println("GBM + FinancialOption Integration Pipeline")
println("="^60)

d_opt = 52
dd_opt = IIDStdUniform(d_opt; seed=7)

Using BrownianMotion (internal GBM construction in FinancialOption)

In [ ]:
tm_bm = BrownianMotion(dd_opt)
f_bm = FinancialOption(tm_bm;
    option_type=:asian, call_put=:call, mean_type=:arithmetic,
    volatility=0.2, start_price=100.0, strike_price=100.0,
    interest_rate=0.05)
sc_bm = CubMCCLT(f_bm; abs_tol=0.1)
result_bm = integrate(sc_bm)
@printf("  Asian Call (via BM):  %.4f  (n = %d)\n", result_bm.solution, result_bm.data[:n])

Using GeometricBrownianMotion directly

In [ ]:
dd_opt2 = IIDStdUniform(d_opt; seed=7)
gbm_tm = GeometricBrownianMotion(dd_opt2;
    t_final=1.0, initial_value=100.0, drift=0.05, diffusion=0.04)
f_gbm = FinancialOption(gbm_tm;
    option_type=:asian, call_put=:call, mean_type=:arithmetic,
    volatility=0.2, start_price=100.0, strike_price=100.0,
    interest_rate=0.05)
sc_gbm = CubMCCLT(f_gbm; abs_tol=0.1)
result_gbm = integrate(sc_gbm)
@printf("  Asian Call (via GBM): %.4f  (n = %d)\n", result_gbm.solution, result_gbm.data[:n])
println()

Different Volatilities

In [ ]:
println("="^60)
println("European Call Prices for Different Volatilities")
println("="^60)

for vol in [0.1, 0.2, 0.3, 0.5, 0.8]
    dd_v = IIDStdUniform(52; seed=7)
    tm_v = BrownianMotion(dd_v)
    f_v = FinancialOption(tm_v;
        option_type=:european, call_put=:call,
        volatility=vol, start_price=100.0, strike_price=100.0,
        interest_rate=0.05)
    sc_v = CubMCCLT(f_v; abs_tol=5.0, n_init=4096)
    result_v = integrate(sc_v)
    @printf("  σ = %.1f: European Call = %.2f\n", vol, result_v.solution)
end
println()

println("="^60)
println("GBM demo completed!")